# OPSD — Single-A100 Training on Google Colab

This notebook runs a **single-GPU** version of [OPSD (On-Policy Self-Distillation)](https://arxiv.org/pdf/2601.18734v3) that follows the paper's **non-thinking Qwen3-1.7B** recipe as closely as one A100 40GB allows, in ~30 min.

The full project targets **4×H100** with vLLM, FlashAttention-2, and DeepSpeed ZeRO-2. This notebook keeps the paper's loss/teacher/LoRA/sampling settings and scales only the compute:

| Setting | Full repo | This notebook |
|---|---|---|
| Model | Qwen3-1.7B / 4B / 8B | **Qwen3-1.7B** |
| GPUs | 4–8 | **1 (A100 40GB)** |
| Generation backend | vLLM (colocate) | **vLLM (colocate)** |
| Attention (training) | `flash_attention_2` | **`sdpa`** |
| Launcher | `accelerate` + DeepSpeed | **plain `python`** |
| Rollout length | 1024 | **1024** |
| Effective batch | 32 | **16** |
| Steps | ~100 (peaks) | **~50** |

> **Before running:** set the runtime to a GPU via **Runtime → Change runtime type → Hardware accelerator: A100 GPU**.

This is a *faithful-settings short run*, not a full reproduction of the paper's AIME numbers (which needs ~100 steps plus a separate 30-50 min evaluation).

## 1. Check the GPU

Confirms a GPU is attached and detects whether it supports `bfloat16` (Ampere+ like A100/L4) or only `float16` (Turing T4).

In [1]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type and select a GPU."
)

gpu_name = torch.cuda.get_device_name(0)
BF16_OK = torch.cuda.is_bf16_supported()
print(f"\nGPU: {gpu_name}")
print(f"bfloat16 supported: {BF16_OK} (otherwise we fall back to float16)")

# Mount Google Drive so checkpoints/outputs persist across runtime resets.
from google.colab import drive
drive.mount("/content/drive")

Mon Jun 15 22:54:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Get the project code into Colab

The next cell sets `PROJECT_DIR` (the folder containing `opsd_train.py`) automatically:

1. **Google Drive (default).** If `DRIVE_PROJECT_DIR` (default `/content/drive/MyDrive/OPSD-main`) exists, it's used directly — no upload needed on reruns. Just put the `OPSD-main` folder in your Drive once.
2. **Already on `/content`.** If you extracted/cloned it earlier this session, it's auto-detected.
3. **Zip upload (fallback).** If neither is found, you'll be prompted to upload a `.zip` of the project.

**Alternative — Clone from GitHub.** If you've pushed the repo, use the commented clone cell below instead.

In [2]:
# Get the project code. Tries, in order:
#   1. Your Drive folder (DRIVE_PROJECT_DIR) — no upload needed on reruns.
#   2. An already-extracted copy anywhere under /content.
#   3. Zip upload as a last resort.
import os, glob, zipfile

# Adjust this if your OPSD folder lives elsewhere on Drive.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/OPSD-main"

def find_project_dir(root="/content"):
    hits = glob.glob(os.path.join(root, "**", "opsd_train.py"), recursive=True)
    return os.path.dirname(hits[0]) if hits else None

if os.path.exists(os.path.join(DRIVE_PROJECT_DIR, "opsd_train.py")):
    PROJECT_DIR = DRIVE_PROJECT_DIR
    print("Using project from Drive.")
else:
    PROJECT_DIR = find_project_dir()

if PROJECT_DIR is None:
    from google.colab import files
    print("Not found on Drive or /content. Upload a .zip of the OPSD project "
          "(the folder containing opsd_train.py)...")
    uploaded = files.upload()
    zip_name = next(n for n in uploaded if n.lower().endswith(".zip"))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content")
    PROJECT_DIR = find_project_dir()

assert PROJECT_DIR, "Could not locate opsd_train.py. Make sure it's on Drive or in the zip."
print("PROJECT_DIR =", PROJECT_DIR)
print(sorted(os.listdir(PROJECT_DIR)))

Using project from Drive.
PROJECT_DIR = /content/drive/MyDrive/OPSD-main
['README.md', '__pycache__', 'accelerate.yaml', 'colab_lightweight_train copy.ipynb', 'data_collator.py', 'environment.yml', 'eval', 'grpo_train.py', 'opsd_train.py', 'opsd_trainer.py', 'scripts', 'sft_train.py']


In [ ]:
# Alternative: clone from GitHub instead of Drive/zip (uncomment + set your URL).
# Drive is already mounted in the GPU cell, so you normally don't need this.
# REPO_URL = "https://github.com/<user>/<repo>.git"
# !git clone --depth 1 $REPO_URL /content/opsd_repo
# import glob, os
# PROJECT_DIR = os.path.dirname(glob.glob('/content/opsd_repo/**/opsd_train.py', recursive=True)[0])
# print('PROJECT_DIR =', PROJECT_DIR)

## 3. Install dependencies

We pin the TRL/Transformers stack to the versions this code was written against, and **add `vllm==0.11.0`** for fast on-policy rollouts (colocate mode) — this is what makes the paper recipe (1024-token generations, ~50 steps) affordable on a single A100 in ~30 min. We still skip `flash-attn`/`deepspeed` (training uses `sdpa`; vLLM handles its own attention).

> The vLLM install is heavier and may take a few minutes. If Colab prompts to **restart the runtime** after install, do it, then re-run from cell 1.

In [3]:
%pip install -q \
    "transformers==4.57.1" \
    "trl==0.26.0" \
    "peft==0.17.1" \
    "accelerate==1.11.0" \
    "datasets==3.6.0" \
    "math-verify==0.8.0" \
    "einops==0.8.1"

# vLLM gives fast on-policy rollouts (colocate mode) — needed to fit the paper
# recipe (1024-token generations, ~50 steps) into ~30 min on a single A100.
# It is installed AFTER the base stack, then transformers/trl are re-pinned so
# vLLM's dependency resolution can't silently downgrade them.
%pip install -q "vllm==0.11.0"
%pip install -q "transformers==4.57.1" "trl==0.26.0"

print("\nDone. If Colab asks you to restart the runtime, do it, then re-run from cell 1.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 139.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.2/517.2 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# Sanity check that the trainer imports without vLLM / flash-attn / deepspeed
import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import importlib
import opsd_trainer  # noqa: F401
importlib.reload(opsd_trainer)
print("opsd_trainer imported OK")

RuntimeError: operator torchvision::nms does not exist

## 4. Configure the lightweight run

These follow the paper's non-thinking Qwen3-1.7B recipe. With more GPU/time you can raise `MAX_STEPS` toward ~100 (where the paper peaks) or increase the effective batch via `PER_DEVICE_BSZ` / `GRAD_ACCUM`. On a smaller GPU, drop `MODEL` to `Qwen/Qwen3-0.6B` and `MAX_COMPLETION` to 256.

In [5]:
# Paper recipe (non-thinking Qwen3-1.7B), scaled to fit one A100 40GB in ~30 min.
MODEL          = "Qwen/Qwen3-1.7B"   # paper headline model (run_opsd_1b.sh)
MAX_STEPS      = 50                    # paper peaks ~50-100 steps; 50 fits the time budget
MAX_COMPLETION = 1024                  # paper rollout length (vLLM makes this affordable)
MAX_LEN        = 4096                  # prompt+completion budget (paper uses 20000 for long thinking traces; non-thinking needs far less)
OUTPUT_DIR     = "/content/drive/MyDrive/opsd_outputs"  # on Drive so checkpoints persist
RUN_NAME       = "colab_a100_nonthink"

# Effective batch = PER_DEVICE_BSZ * GRAD_ACCUM on this single GPU (paper: 32 over 4 GPUs).
# The OPSD JSD loss materializes FULL-vocab logits (~152k) for student AND teacher,
# so per-step memory is dominated by PER_DEVICE_BSZ. Keep it at 1 on a 40GB A100 and
# get the batch from GRAD_ACCUM instead. Raise PER_DEVICE_BSZ only on bigger GPUs.
PER_DEVICE_BSZ = 1
GRAD_ACCUM     = 16

# Fraction of GPU memory reserved for the colocated vLLM engine. The rest is left for
# training (model + activations + full-vocab logits). Lower this if you hit OOM.
VLLM_UTIL      = 0.45

DTYPE = "bfloat16" if BF16_OK else "float16"
# In transformers 4.57, --bf16/--fp16 require an explicit value (e.g. `--bf16 True`),
# not a bare flag, so we pass `True` here.
PREC_FLAG = "--bf16 True" if BF16_OK else "--fp16 True"
print(f"Model={MODEL}  dtype={DTYPE} ({PREC_FLAG})  eff_batch={PER_DEVICE_BSZ * GRAD_ACCUM}  steps={MAX_STEPS}")

Model=Qwen/Qwen3-1.7B  dtype=bfloat16 (--bf16 True)  eff_batch=16  steps=50


## 5. Run training

Runs `opsd_train.py` as a plain single-GPU process, following the paper's **non-thinking Qwen3-1.7B** recipe (`run_opsd_4b_nonthink.sh` flags, 1.7B model from `run_opsd_1b.sh`), scaled to one A100 40GB:
- `--use_peft --fixed_teacher` with `--lora_r 64 --lora_alpha 128`: LoRA student with a frozen base-model teacher (paper's main setup).
- `--use_vllm --vllm_mode colocate --vllm_gpu_memory_utilization 0.6`: fast on-policy rollouts in the same process (the key to fitting 1024-token generations in the time budget).
- `--beta 0 --lmbda 1 --jsd_token_clip 1e-6`, `--temperature 1.1 --top_p 0.95 --top_k 20`: paper loss + sampling.
- `--student_thinking False --teacher_thinking False`: non-thinking mode (faster than thinking).
- `--learning_rate 5e-6 --max_grad_norm 0.1`, effective batch 16 (vs paper's 32 over 4 GPUs), `--gradient_checkpointing` for memory.
- WandB disabled.

**Expectations:** the first step is slow (model + 30k dataset download, vLLM engine init). After that, vLLM rollouts are fast; ~50 steps should land around ~20-30 min. This is a faithful-settings short run, **not** a full reproduction of the paper's AIME numbers (that needs ~100 steps + a separate 30-50 min eval). If you hit CUDA OOM, lower `PER_DEVICE_BSZ`, `--vllm_gpu_memory_utilization`, or `MAX_COMPLETION` in the config cell.

In [ ]:
import subprocess, shlex, sys

cmd = f"""{sys.executable} -u opsd_train.py \
  --model_name_or_path {MODEL} \
  --attn_implementation sdpa \
  --torch_dtype {DTYPE} {PREC_FLAG} \
  --learning_rate 5e-6 \
  --max_grad_norm 0.1 \
  --per_device_train_batch_size {PER_DEVICE_BSZ} \
  --gradient_accumulation_steps {GRAD_ACCUM} \
  --gradient_checkpointing \
  --output_dir {OUTPUT_DIR} \
  --run_config {RUN_NAME} \
  --max_steps {MAX_STEPS} \
  --num_train_epochs 30 \
  --max_completion_length {MAX_COMPLETION} \
  --max_length {MAX_LEN} \
  --save_steps 25 \
  --logging_steps 1 \
  --beta 0 --lmbda 1 \
  --temperature 1.1 --top_p 0.95 --top_k 20 \
  --use_vllm --vllm_mode colocate --vllm_gpu_memory_utilization {VLLM_UTIL} --vllm_tensor_parallel_size 1 \
  --use_peft --lora_r 64 --lora_alpha 128 \
  --lora_target_modules q_proj k_proj v_proj o_proj gate_proj up_proj down_proj \
  --fixed_teacher \
  --student_thinking False --teacher_thinking False \
  --jsd_token_clip 1e-6 \
  --report_to none"""

env = dict(os.environ)
env.update({
    "WANDB_MODE": "disabled",
    "WANDB_DISABLED": "true",
    "TOKENIZERS_PARALLELISM": "false",
    # Reduce CUDA memory fragmentation (recommended in the OOM traceback).
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})

print(cmd, "\n\n--- launching ---\n")
# Stream output live AND capture it so errors are visible (subprocess.run with no
# capture can swallow the traceback in Colab, leaving only "Exit code: 2").
proc = subprocess.Popen(
    shlex.split(cmd),
    cwd=PROJECT_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
output_lines = []
for line in proc.stdout:
    print(line, end="")
    output_lines.append(line)
proc.wait()
print("\nExit code:", proc.returncode)

/usr/bin/python3 -u opsd_train.py   --model_name_or_path Qwen/Qwen3-1.7B   --attn_implementation sdpa   --torch_dtype bfloat16 --bf16 True   --learning_rate 5e-6   --max_grad_norm 0.1   --per_device_train_batch_size 1   --gradient_accumulation_steps 16   --gradient_checkpointing   --output_dir /content/drive/MyDrive/opsd_outputs   --run_config colab_a100_nonthink   --max_steps 50   --num_train_epochs 30   --max_completion_length 1024   --max_length 4096   --save_steps 25   --logging_steps 1   --beta 0 --lmbda 1   --temperature 1.1 --top_p 0.95 --top_k 20   --use_vllm --vllm_mode colocate --vllm_gpu_memory_utilization 0.45 --vllm_tensor_parallel_size 1   --use_peft --lora_r 64 --lora_alpha 128   --lora_target_modules q_proj k_proj v_proj o_proj gate_proj up_proj down_proj   --fixed_teacher   --student_thinking False --teacher_thinking False   --jsd_token_clip 1e-6   --report_to none 

--- launching ---

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in

## 6. Inspect outputs & quick generation test

The trained LoRA adapter is saved under `OUTPUT_DIR/RUN_NAME`. Below we load the base model + adapter and generate a short answer to a math prompt to confirm the checkpoint is usable.

In [ ]:
import os
adapter_dir = os.path.join(OUTPUT_DIR, RUN_NAME)
print("Adapter dir:", adapter_dir)
print(sorted(os.listdir(adapter_dir)) if os.path.isdir(adapter_dir) else "(not found)")

Adapter dir: /content/drive/MyDrive/opsd_outputs/colab_lightweight
(not found)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

dtype = torch.bfloat16 if BF16_OK else torch.float16
tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype, attn_implementation="sdpa").cuda()
model = PeftModel.from_pretrained(base, adapter_dir).cuda().eval()

messages = [{"role": "user", "content": "What is 17 * 24? Show your reasoning briefly."}]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tok(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Next steps

- **Closer to the paper:** raise `MAX_STEPS` toward ~100 (where Qwen3-1.7B peaks) and the effective batch toward 32 — needs more time/GPU than a single 30-min A100 session.
- **Thinking mode:** for the headline AIME numbers, set `--student_thinking True --teacher_thinking True` and a larger `MAX_LEN` (the paper uses 20000 for long traces); much slower per step.
- **Real evaluation:** run `eval/run_eval_nonthink.sh` (or `run_eval.sh` for thinking) on saved checkpoints — this is a separate ~30-50 min job, not part of training.
- **Multi-GPU / DeepSpeed:** use the original `scripts/run_opsd_*.sh` with `accelerate launch --config_file accelerate.yaml` on a proper multi-GPU machine.
- **Persisting results:** checkpoints already save to your Drive `OUTPUT_DIR` so they survive runtime resets.